# Tâche 2 — Exploration & Analyse (Notebook)
**Auteur : Kakpo Christamour**

Notebook daté 05/10 reprenant le script `Tache2_ANIP_WPP.py`. Il produit le dataset enrichi, détecte anomalies et exporte graphiques.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
print('Python', sys.version)
print('pandas', pd.__version__)

In [ ]:
# Chemins et création des dossiers
input_file = 'outputs/tache1/WPP2024_DEMOGRAPHIC_CLEAN.csv'
output_file = 'outputs/tache2/WPP2024_DEMOGRAPHIC_ENRICHED.csv'
anomalies_file = 'outputs/tache2/WPP2024_DEMOGRAPHIC_ANOMALIES.csv'
graphs_dir = 'outputs/tache2/graphs'
os.makedirs('outputs/tache2', exist_ok=True)
os.makedirs(graphs_dir, exist_ok=True)
print('Input:', input_file)
print('Output:', output_file)

In [ ]:
# Charger le dataset
df = pd.read_csv(input_file)
print('Dataset chargé :', df.shape)

In [ ]:
# Remplacer valeurs manquantes et nettoyage
fill_cols = ['population_total','population_male','population_female','pop_growth_rate','sex_ratio','regional_dev_index','life_expectancy','fertility_rate','birth_rate','death_rate']
for col in fill_cols:
    if col not in df.columns:
        df[col] = 0
df = df[df['population_total'] > 0]
print('Après nettoyage :', df.shape)

In [ ]:
# Détection d'anomalies
missing = df.isna().sum()
print('Valeurs manquantes :', missing[missing>0] if missing.any() else 'Aucune')
df['pop_diff'] = df.groupby('region')['population_total'].diff()
anomalies_pop = df[df['pop_diff'] < 0]
print('Populations décroissantes suspectes :', anomalies_pop.shape[0])
anomalies = pd.concat([anomalies_pop]).drop_duplicates()
anomalies.to_csv(anomalies_file, index=False)
print('Anomalies sauvegardées :', anomalies_file)

In [ ]:
# Créer nouvelles variables
df['pop_growth_rate'] = df.groupby('region')['population_total'].pct_change() * 100
df['sex_ratio'] = df.apply(lambda row: row['population_male']/row['population_female'] if row['population_female']>0 else 1, axis=1)
components = []
if 'median_age' in df.columns:
    components.append(df['median_age'] / df['median_age'].max())
if 'birth_rate' in df.columns:
    components.append(1 - df['birth_rate'] / df['birth_rate'].max())
if 'life_expectancy' in df.columns:
    components.append(df['life_expectancy'] / df['life_expectancy'].max())
df['regional_dev_index'] = sum(components)/len(components) if components else 0
df['world_population'] = df.groupby('year')['population_total'].transform('sum')
df['population_pct'] = df['population_total'] / df['world_population'] * 100
print('Nouvelles variables créées')

In [ ]:
# Agrégation region x year
agg_cols = ['population_total','population_male','population_female','pop_growth_rate','sex_ratio','regional_dev_index','population_pct']
df_agg = df.groupby(['region','year']).agg({**{col:'mean' for col in agg_cols}, 'iso3':'first'}).reset_index()
print('Agrégation:', df_agg.shape)

In [ ]:
# Graphiques exploratoires
plt.figure(figsize=(10,5))
df.groupby('year')['population_total'].sum().plot()
plt.title('Population mondiale (1950-2023)')
plt.ylabel('Population (milliers)')
plt.savefig(f'{graphs_dir}/population_mondiale.png')
plt.close()
plt.figure(figsize=(8,6))
sns.heatmap(df_agg[['population_total','pop_growth_rate','sex_ratio','regional_dev_index']].corr(), annot=True, cmap='coolwarm')
plt.title('Corrélations principales')
plt.savefig(f'{graphs_dir}/correlation.png')
plt.close()
print('Graphiques sauvegardés dans', graphs_dir)

In [ ]:
# Export dataset enrichi
df_agg.to_csv(output_file, index=False)
print('Dataset enrichi sauvegardé :', output_file)